# DS102 V27 — ứng viên thử nghiệm, không pretrained
Mốc Kaggle trước đó:0.77436. OOF V27 tăng0.000925, dưới ngưỡng chọn chính thức0.001; không gọi là mô hình đã vượt mọi gate.
CSV thử nghiệm chỉ được xuất sau full-fit khi dev không giảm. Chưa xác nhận mức tăng Kaggle; không tự submit.
Notebook fit lại toàn bộ pipeline từ train trên hosted Colab, không cần cache dự đoán; dev giữ riêng, chỉ dùng bốn CSV được cung cấp.
OOF/dev được xem nhiều lần nên còn thiên lệch lựa chọn. Source đầy đủ được nhúng; scikit-learn BSD-3-Clause, CatBoost Apache-2.0.


In [ ]:
import os,platform,sys,subprocess
assert platform.system()=='Linux' and os.environ.get('COLAB_RELEASE_TAG'),'Hosted Colab required'
subprocess.check_call([sys.executable,'-m','pip','install','-q','numpy==2.4.6','scipy==1.16.3','pandas==2.2.3','scikit-learn==1.7.2','catboost==1.2.10'])


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded=files.upload()
data_dir=Path('/content/ds102_v27_data');data_dir.mkdir(exist_ok=True)
required={'ds102_train.csv','ds102_dev.csv','ds102_test.csv','ds102_samplesubmission.csv'}
for name,content in uploaded.items():
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            for member in z.namelist():
                if Path(member).name in required:
                    (data_dir/Path(member).name).write_bytes(z.read(member))
    elif Path(name).name in required:
        (data_dir/Path(name).name).write_bytes(content)
assert all((data_dir/n).exists() for n in required),'Missing competition CSV'
del uploaded

import hashlib
expected_data_sha256={'ds102_train.csv': '15202b19b2dc2c6c89afeb209024bb0dc77ddf927de44381a96c24a961d823a5', 'ds102_dev.csv': 'd895cc298b1e5b56bbb95c9f9b15610e5207a00cfd12cc79f32061e27b7a8ccf', 'ds102_test.csv': '82c21afd6f8f4610912bc12ed19c3b6b1a586fb0dfa5d2296f6548ae4fc6b398', 'ds102_samplesubmission.csv': '4cf7832636057152395b8b20302c11a691fd2ab35d50057ef43443f3d388424d'}
for name,h in expected_data_sha256.items():
    assert hashlib.sha256((data_dir/name).read_bytes()).hexdigest()==h, "Competition CSV differs: "+name


In [ ]:
import json
from pathlib import Path
source=Path("/content/ds102_v27_experimental_source");source.mkdir(exist_ok=True)
modules=json.loads('{"train_submit.py": "\\"\\"\\"CPU baseline cho DS102: TF-IDF + Linear SVM, chọn theo F1-macro.\\n\\nChỉ dùng comment và nhãn train. Dev được đánh giá sau khi chọn tham số\\nbằng cross-validation trên train; không gộp dev vào train.\\n\\"\\"\\"\\n\\nimport argparse\\nimport html\\nimport json\\nimport platform\\nimport re\\nimport unicodedata\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nimport sklearn\\nfrom scipy import sparse\\nfrom sklearn.base import BaseEstimator, TransformerMixin\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom sklearn.pipeline import FeatureUnion, Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.svm import LinearSVC\\n\\n\\nSEED = 102\\nLABELS = [0, 1, 2, 3]\\nFILES = {name: f\\"ds102_{name}.csv\\" for name in [\\"train\\", \\"dev\\", \\"test\\", \\"samplesubmission\\"]}\\n\\n\\ndef normalize_text(value):\\n    \\"\\"\\"Giữ dấu, emoji, dấu câu và ký tự lặp vì có thể giúp nhận diện spam.\\"\\"\\"\\n    text = html.unescape(unicodedata.normalize(\\"NFC\\", str(value)))\\n    text = text.replace(\\"\\\\\\\\r\\\\\\\\n\\", \\"\\\\n\\").replace(\\"\\\\\\\\n\\", \\"\\\\n\\").replace(\\"\\\\\\\\r\\", \\"\\\\n\\")\\n    return re.sub(r\\"\\\\s+\\", \\" \\", text).strip().lower()\\n\\n\\nclass ReviewStats(BaseEstimator, TransformerMixin):\\n    \\"\\"\\"Đặc trưng độ dài, lặp ký tự và tỷ lệ ký tự; không có từ khóa nhãn.\\"\\"\\"\\n\\n    def fit(self, X, y=None):\\n        return self\\n\\n    def transform(self, X):\\n        rows = []\\n        for text in X:\\n            chars = len(text)\\n            words = text.split()\\n            nwords = len(words)\\n            denom = max(chars, 1)\\n            word_denom = max(nwords, 1)\\n            letters = [c for c in text if c.isalpha()]\\n            longest_run = max((len(m.group()) for m in re.finditer(r\\"(.)\\\\1*\\", text)), default=0)\\n            rows.append([\\n                np.log1p(chars), np.log1p(nwords), chars / word_denom,\\n                len(set(words)) / word_denom, len(set(text)) / denom,\\n                longest_run / denom, np.log1p(longest_run),\\n                sum(c.isdigit() for c in text) / denom,\\n                len(letters) / denom,\\n                sum(not c.isalnum() and not c.isspace() for c in text) / denom,\\n                sum(ord(c) > 0xFFFF for c in text) / denom,\\n                text.count(\\"!\\") / denom, text.count(\\"?\\") / denom,\\n                float(bool(re.search(r\\"https?://|www\\\\.\\", text))),\\n            ])\\n        return sparse.csr_matrix(np.asarray(rows, dtype=np.float32))\\n\\n\\ndef make_features():\\n    return FeatureUnion([\\n        (\\"word\\", TfidfVectorizer(\\n            ngram_range=(1, 2), min_df=2, max_df=1.0, max_features=90000,\\n            sublinear_tf=True, lowercase=False, dtype=np.float32,\\n            token_pattern=r\\"(?u)\\\\b\\\\w+\\\\b\\",\\n        )),\\n        (\\"char\\", TfidfVectorizer(\\n            analyzer=\\"char\\", ngram_range=(2, 5), min_df=3, max_features=160000,\\n            sublinear_tf=True, lowercase=False, dtype=np.float32,\\n        )),\\n        (\\"stats\\", Pipeline([\\n            (\\"extract\\", ReviewStats()), (\\"scale\\", StandardScaler(with_mean=False)),\\n        ])),\\n    ], transformer_weights={\\"word\\": 1.0, \\"char\\": 1.0, \\"stats\\": 0.1})\\n\\n\\ndef make_model(config, y):\\n    weight = config[\\"weight\\"]\\n    if weight == \\"sqrt_balanced\\":\\n        counts = pd.Series(y).value_counts()\\n        weight = {int(k): float(np.sqrt(len(y) / (len(counts) * v))) for k, v in counts.items()}\\n    elif weight == \\"none\\":\\n        weight = None\\n    return LinearSVC(C=config[\\"C\\"], class_weight=weight, dual=True,\\n                     max_iter=10000, tol=1e-4, random_state=SEED)\\n\\n\\ndef find_data_dir(supplied=None):\\n    if supplied:\\n        path = Path(supplied).expanduser().resolve()\\n        if all((path / f).is_file() for f in FILES.values()):\\n            return path\\n        raise FileNotFoundError(f\\"Thiếu một trong 4 file CSV tại {path}\\")\\n    candidates = [Path.cwd(), Path.cwd() / \\"data\\", Path.cwd() / \\"dataset\\"]\\n    kaggle = Path(\\"/kaggle/input\\")\\n    if kaggle.exists():\\n        candidates += sorted({p.parent for p in kaggle.rglob(FILES[\\"train\\"])})\\n    matches = [p for p in candidates if all((p / f).is_file() for f in FILES.values())]\\n    if len(matches) == 1:\\n        return matches[0].resolve()\\n    raise FileNotFoundError(\\"Hãy truyền --data-dir đến thư mục chứa đúng 4 CSV; trên Kaggle dùng Add Input trước.\\")\\n\\n\\ndef load_data(data_dir):\\n    data = {name: pd.read_csv(data_dir / file, keep_default_na=False) for name, file in FILES.items()}\\n    for name in [\\"train\\", \\"dev\\", \\"test\\"]:\\n        required = {\\"comment\\", \\"spam_label\\"} if name != \\"test\\" else {\\"comment\\", \\"id\\"}\\n        if not required.issubset(data[name].columns):\\n            raise ValueError(f\\"{name}: thiếu cột {required - set(data[name].columns)}\\")\\n        if data[name].empty:\\n            raise ValueError(f\\"{name} rỗng\\")\\n    for name in [\\"train\\", \\"dev\\"]:\\n        values = pd.to_numeric(data[name][\\"spam_label\\"], errors=\\"raise\\")\\n        if not values.isin(LABELS).all():\\n            raise ValueError(f\\"{name}: nhãn phải thuộc {LABELS}\\")\\n        data[name][\\"spam_label\\"] = values.astype(int)\\n    if set(data[\\"train\\"][\\"spam_label\\"]) != set(LABELS):\\n        raise ValueError(\\"Train phải có đủ 4 nhãn\\")\\n    sample = data[\\"samplesubmission\\"]\\n    if list(sample.columns) != [\\"ID\\", \\"spam_label\\"]:\\n        raise ValueError(\\"Sample phải có hai cột ID,spam_label\\")\\n    test_ids, sample_ids = data[\\"test\\"][\\"id\\"].astype(str), sample[\\"ID\\"].astype(str)\\n    if not test_ids.is_unique or not sample_ids.is_unique or set(test_ids) != set(sample_ids):\\n        raise ValueError(\\"ID trong test và sample phải duy nhất và trùng tập hợp\\")\\n    return data\\n\\n\\ndef metrics(y, pred):\\n    return {\\n        \\"macro_f1\\": float(f1_score(y, pred, labels=LABELS, average=\\"macro\\", zero_division=0)),\\n        \\"accuracy\\": float(accuracy_score(y, pred)),\\n        \\"classification_report\\": classification_report(y, pred, labels=LABELS, output_dict=True, zero_division=0),\\n        \\"confusion_matrix\\": confusion_matrix(y, pred, labels=LABELS).tolist(),\\n    }\\n\\n\\ndef main(argv=None):\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--data-dir\\", default=None)\\n    parser.add_argument(\\"--output-dir\\", default=\\"output\\")\\n    parser.add_argument(\\"--folds\\", type=int, default=3)\\n    args = parser.parse_args(argv)\\n    if args.folds < 2:\\n        parser.error(\\"--folds phải >= 2\\")\\n    data_dir = find_data_dir(args.data_dir)\\n    output = Path(args.output_dir).expanduser().resolve()\\n    output.mkdir(parents=True, exist_ok=True)\\n    data = load_data(data_dir)\\n    train, dev, test = [data[n] for n in [\\"train\\", \\"dev\\", \\"test\\"]]\\n    texts = train[\\"comment\\"].map(normalize_text).to_numpy()\\n    dev_texts = dev[\\"comment\\"].map(normalize_text).to_numpy()\\n    test_texts = test[\\"comment\\"].map(normalize_text).to_numpy()\\n    y = train[\\"spam_label\\"].to_numpy()\\n    if train.groupby(train[\\"comment\\"].map(normalize_text))[\\"spam_label\\"].first().value_counts().min() < args.folds:\\n        raise ValueError(\\"Không đủ nhóm cho số folds đã chọn\\")\\n    print(f\\"Train={len(train)}, dev={len(dev)}, test={len(test)}\\", flush=True)\\n    print(f\\"Nhãn train: {train.spam_label.value_counts().sort_index().to_dict()}\\", flush=True)\\n    configs = [{\\"C\\": c, \\"weight\\": weight} for weight in [\\"none\\", \\"sqrt_balanced\\", \\"balanced\\"] for c in [1.0, 4.0]]\\n    oof = np.full((len(configs), len(y)), -1, dtype=np.int64)\\n    fold_scores = [[] for _ in configs]\\n    splitter = StratifiedGroupKFold(n_splits=args.folds, shuffle=True, random_state=SEED)\\n    for fold, (fit_idx, val_idx) in enumerate(splitter.split(texts, y, groups=texts), 1):\\n        print(f\\"Fold {fold}/{args.folds}: fit TF-IDF chỉ trên train của fold...\\", flush=True)\\n        features = make_features()\\n        X_fit = features.fit_transform(texts[fit_idx])\\n        X_val = features.transform(texts[val_idx])\\n        for i, config in enumerate(configs):\\n            model = make_model(config, y[fit_idx])\\n            model.fit(X_fit, y[fit_idx])\\n            pred = model.predict(X_val)\\n            oof[i, val_idx] = pred\\n            score = f1_score(y[val_idx], pred, labels=LABELS, average=\\"macro\\", zero_division=0)\\n            fold_scores[i].append(float(score))\\n            print(f\\"  {config}: F1-macro={score:.5f}\\", flush=True)\\n        del X_fit, X_val, features\\n    cv_scores = [float(f1_score(y, p, labels=LABELS, average=\\"macro\\", zero_division=0)) for p in oof]\\n    best_idx = int(np.argmax(cv_scores))\\n    best = configs[best_idx]\\n    print(f\\"Chọn bằng OOF train: {best}, F1-macro={cv_scores[best_idx]:.5f}\\", flush=True)\\n    print(\\"Huấn luyện cấu hình đã chọn trên toàn bộ train...\\", flush=True)\\n    features = make_features()\\n    X_train = features.fit_transform(texts)\\n    model = make_model(best, y)\\n    model.fit(X_train, y)\\n    dev_pred = model.predict(features.transform(dev_texts))\\n    dev_y = dev[\\"spam_label\\"].to_numpy()\\n    report = metrics(dev_y, dev_pred)\\n    unique_dev = ~pd.Series(dev_texts).isin(set(texts)).to_numpy()\\n    print(f\\"Dev F1-macro={report[\'macro_f1\']:.5f}, accuracy={report[\'accuracy\']:.5f}\\", flush=True)\\n    print(classification_report(dev_y, dev_pred, labels=LABELS, digits=4, zero_division=0), flush=True)\\n    print(f\\"Dev có {int((~unique_dev).sum())} comment chuẩn hóa xuất hiện trong train.\\", flush=True)\\n    test_pred = model.predict(features.transform(test_texts)).astype(int)\\n    by_id = pd.Series(test_pred, index=test[\\"id\\"].astype(str))\\n    submission = data[\\"samplesubmission\\"].copy()\\n    submission[\\"spam_label\\"] = submission[\\"ID\\"].astype(str).map(by_id)\\n    if submission[\\"spam_label\\"].isna().any() or not submission[\\"spam_label\\"].isin(LABELS).all():\\n        raise ValueError(\\"Submission có nhãn thiếu hoặc không hợp lệ\\")\\n    submission[\\"spam_label\\"] = submission[\\"spam_label\\"].astype(int)\\n    submission_path = output / \\"submission.csv\\"\\n    submission.to_csv(submission_path, index=False)\\n    saved = pd.read_csv(submission_path)\\n    pd.testing.assert_frame_equal(saved, submission, check_dtype=False)\\n    if len(saved) != len(test) or not saved[\\"ID\\"].equals(data[\\"samplesubmission\\"][\\"ID\\"]):\\n        raise ValueError(\\"Sai số hàng hoặc thứ tự ID khi lưu\\")\\n    detail = pd.DataFrame({\\"comment\\": dev[\\"comment\\"], \\"true_label\\": dev_y, \\"predicted_label\\": dev_pred,\\n                           \\"comment_seen_in_train\\": ~unique_dev})\\n    detail.to_csv(output / \\"dev_predictions.csv\\", index=False)\\n    full_report = {\\n        \\"seed\\": SEED, \\"python\\": platform.python_version(), \\"sklearn\\": sklearn.__version__,\\n        \\"rows\\": {k: len(v) for k, v in data.items()}, \\"feature_columns\\": [\\"comment\\"],\\n        \\"selection\\": \\"StratifiedGroupKFold on train; normalized comments as groups; OOF macro F1\\",\\n        \\"fit_data\\": \\"train only; dev and test never used to fit TF-IDF, scaler or SVM\\",\\n        \\"candidates\\": [{**c, \\"fold_macro_f1\\": f, \\"oof_macro_f1\\": s}\\n                       for c, f, s in zip(configs, fold_scores, cv_scores)],\\n        \\"chosen_config\\": best, \\"dev\\": report,\\n        \\"dev_comments_seen_in_train\\": int((~unique_dev).sum()),\\n        \\"dev_without_comment_overlap\\": metrics(dev_y[unique_dev], dev_pred[unique_dev]) if unique_dev.any() else None,\\n        \\"test_prediction_counts\\": {int(k): int(v) for k, v in submission.spam_label.value_counts().items()},\\n        \\"submission_rows_verified\\": len(saved),\\n    }\\n    (output / \\"metrics.json\\").write_text(json.dumps(full_report, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    print(f\\"Đã kiểm tra và lưu {len(saved)} dự đoán: {submission_path}\\", flush=True)\\n    return full_report\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "improved_features.py": "\\"\\"\\"Đặc trưng v2: văn bản review và mức trùng với thông tin sản phẩm.\\"\\"\\"\\nimport re\\nfrom collections import Counter\\nimport numpy as np\\nfrom scipy import sparse\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.preprocessing import StandardScaler, OneHotEncoder\\nfrom train_submit import normalize_text, ReviewStats\\n\\n\\ndef prepare(frame):\\n    frame = frame.copy()\\n    for col in [\'comment\', \'product_name\', \'description\', \'category\', \'categories\']:\\n        frame[col] = frame[col].fillna(\'\').astype(str).map(normalize_text)\\n    return frame\\n\\n\\ndef tokens(text):\\n    return re.findall(r\'(?u)\\\\b\\\\w+\\\\b\', text)\\n\\n\\ndef ngrams(words, n):\\n    return set(zip(*(words[i:] for i in range(n))))\\n\\n\\ndef dense_features(df):\\n    basic = ReviewStats().transform(df.comment).toarray()\\n    extra = []\\n    for row in df.itertuples(index=False):\\n        text = row.comment\\n        words = tokens(text)\\n        wl = [len(w) for w in words] or [0]\\n        wc = Counter(words)\\n        n = max(len(words), 1)\\n        feats = [np.log1p(max(wl)), np.mean(wl), np.std(wl),\\n                 sum(len(w) > 12 for w in words) / n,\\n                 max(wc.values(), default=0) / n,\\n                 sum(v * (v - 1) for v in wc.values()) / max(n * (n - 1), 1),\\n                 text.count(\'.\') / max(len(text), 1),\\n                 len(re.findall(r\'(.)\\\\1{3,}\', text)) / n,\\n                 np.log1p(len(text.split())),\\n                 float(row.rating) / 5,\\n                 np.log1p(max(float(row.num_sold), 0)),\\n                 np.log1p(max(float(row.num_review), 0))]\\n        for context in [row.product_name, row.description]:\\n            context_words = tokens(context)\\n            for size in [1, 2, 3, 4]:\\n                a, b = ngrams(words, size), ngrams(context_words, size)\\n                overlap = len(a & b)\\n                ratio = overlap / max(len(a), 1)\\n                feats += [ratio, ratio ** 2, ratio * np.log1p(len(words)), np.log1p(overlap)]\\n            feats += [float(len(text) > 10 and text in context), np.log1p(len(context))]\\n        extra.append(feats)\\n    return np.hstack([basic, np.asarray(extra, dtype=np.float32)])\\n\\n\\nclass FeaturesV2:\\n    def __init__(self):\\n        self.word = TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_features=140000,\\n                                   sublinear_tf=True, lowercase=False, dtype=np.float32,\\n                                   token_pattern=r\'(?u)\\\\b\\\\w+\\\\b\')\\n        self.char = TfidfVectorizer(analyzer=\'char\', ngram_range=(2, 5), min_df=3,\\n                                   max_features=180000, sublinear_tf=True,\\n                                   lowercase=False, dtype=np.float32)\\n        self.product = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=20000,\\n                                      sublinear_tf=True, lowercase=False, dtype=np.float32)\\n        self.category = OneHotEncoder(handle_unknown=\'ignore\', dtype=np.float32)\\n        self.scale = StandardScaler()\\n\\n    def fit_transform(self, df):\\n        text = sparse.hstack([self.word.fit_transform(df.comment), self.char.fit_transform(df.comment)], format=\'csr\')\\n        meta = sparse.hstack([self.product.fit_transform(df.product_name) * .2,\\n                             self.category.fit_transform(df[[\'category\', \'rating\']].astype(str)) * .2,\\n                             sparse.csr_matrix(self.scale.fit_transform(dense_features(df))) * .15], format=\'csr\')\\n        return text, meta\\n\\n    def transform(self, df):\\n        text = sparse.hstack([self.word.transform(df.comment), self.char.transform(df.comment)], format=\'csr\')\\n        meta = sparse.hstack([self.product.transform(df.product_name) * .2,\\n                             self.category.transform(df[[\'category\', \'rating\']].astype(str)) * .2,\\n                             sparse.csr_matrix(self.scale.transform(dense_features(df))) * .15], format=\'csr\')\\n        return text, meta\\n\\n\\ndef nb_ratios(X, y):\\n    \\"\\"\\"Log-count ratio one-vs-rest, fit chỉ từ phần train của fold.\\"\\"\\"\\n    ratios = []\\n    for c in range(4):\\n        pos = np.asarray(X[y == c].sum(axis=0)).ravel() + 1\\n        neg = np.asarray(X[y != c].sum(axis=0)).ravel() + 1\\n        ratios.append(np.log((pos / pos.sum()) / (neg / neg.sum())))\\n    return ratios\\n", "select_v2.py": "\\"\\"\\"Chọn mô hình/ensemble và bias bằng OOF train, không xem nhãn dev.\\"\\"\\"\\nimport json\\nfrom pathlib import Path\\nimport numpy as np\\n\\n\\ndef macro(y, pred):\\n    cm = np.bincount(4 * y + pred, minlength=16).reshape(4, 4)\\n    den = cm.sum(axis=0) + cm.sum(axis=1)\\n    return float(np.mean(np.divide(2 * np.diag(cm), den, out=np.zeros(4), where=den > 0)))\\n\\n\\ndef calibrate(scores, y):\\n    # 3 offsets tương đối so với nhãn 0; grid nhỏ để hạn chế overfit.\\n    bias = np.zeros(4)\\n    for _ in range(3):\\n        previous = bias.copy()\\n        for cls in [1, 2, 3]:\\n            choices = []\\n            for value in np.linspace(-.4, .4, 17):\\n                proposal = bias.copy()\\n                proposal[cls] = value\\n                objective = macro(y, (scores + proposal).argmax(axis=1)) - .005 * np.square(proposal).sum()\\n                choices.append((objective, -abs(value), float(value)))\\n            bias[cls] = max(choices)[2]\\n        if np.allclose(previous, bias):\\n            break\\n    return bias\\n\\n\\ndef select(scores, y, folds, configs):\\n    # NB-SVM C=1/2 có cảnh báo chưa hội tụ ở thử nghiệm, không chọn chúng.\\n    candidates = [(cfg[\'name\'], {i: 1.}) for i, cfg in enumerate(configs)\\n                  if cfg[\'kind\'] != \'nb\' or cfg[\'C\'] == .5]\\n    # Blends định trước, tránh tìm hàng trăm trọng số trên lớp hiếm.\\n    byname = {c[\'name\']: i for i, c in enumerate(configs)}\\n    for svm_w in [.25, .5, .75]:\\n        candidates.append((f\'blend_svm_nb_{svm_w}\',\\n                           {byname[\'svm_0.5_balanced\']: svm_w, byname[\'nb_0.5\']: 1-svm_w}))\\n    rows = []\\n    for name, weights in candidates:\\n        s = sum(scores[i] * w for i, w in weights.items())\\n        cross_pred = np.zeros(len(y), dtype=int)\\n        for f in np.unique(folds):\\n            fit = folds != f\\n            b = calibrate(s[fit], y[fit])\\n            cross_pred[~fit] = (s[~fit] + b).argmax(axis=1)\\n        raw, calibrated = macro(y, s.argmax(axis=1)), macro(y, cross_pred)\\n        # Chọn bias chỉ nếu calibration cải thiện khi kiểm tra chéo.\\n        use_bias = calibrated > raw + .002\\n        row = {\'name\': name, \'weights\': weights, \'raw_oof_macro_f1\': raw,\\n               \'cross_calibration_macro_f1\': calibrated, \'use_bias\': use_bias,\\n               \'selection_score\': calibrated if use_bias else raw,\\n               \'bias\': calibrate(s, y).tolist() if use_bias else [0., 0., 0., 0.]}\\n        rows.append(row)\\n    rows.sort(key=lambda r: r[\'selection_score\'], reverse=True)\\n    return rows\\n\\n\\nif __name__ == \'__main__\':\\n    folder = Path(\'kaggle_ds102/experiments_v2\')\\n    z = np.load(folder/\'oof.npz\')\\n    cfg = json.loads((folder/\'configs.json\').read_text())\\n    rows = select(z[\'scores\'], z[\'y\'], z[\'folds\'], cfg)\\n    (folder/\'selection.json\').write_text(json.dumps(rows, indent=2))\\n    for r in rows:\\n        print(r[\'name\'], \'raw\', round(r[\'raw_oof_macro_f1\'], 5),\\n              \'cross calibrated\', round(r[\'cross_calibration_macro_f1\'], 5), \'bias\', r[\'bias\'])\\n", "selection_v5.py": "\\"\\"\\"Coarse probability reweighting, validated on separate OOF folds.\\"\\"\\"\\nfrom itertools import product\\nimport numpy as np\\nfrom select_v2 import macro\\n\\n\\ndef fit_weights(probabilities, y):\\n    choices = []\\n    # A small fixed grid: minority-class recall matters for macro F1.\\n    # Identity is included. Never access development or test labels here.\\n    for a,b,c in product([.8,1.,1.2,1.4], [.8,1.,1.2,1.4], [.9,1.,1.1]):\\n        w=np.array([1.,a,b,c])\\n        score=macro(y,(probabilities*w).argmax(1))\\n        penalty=.002*float(np.square(np.log(w)).sum())\\n        choices.append((score-penalty,-float(np.square(np.log(w)).sum()),w))\\n    return max(choices,key=lambda row:row[:2])[2]\\n\\n\\ndef cross_weight(probabilities,y,folds):\\n    pred=np.empty(len(y),dtype=int); weights=[]; fold_deltas=[]\\n    for fold in np.unique(folds):\\n        fit=folds!=fold\\n        w=fit_weights(probabilities[fit],y[fit]); weights.append(w.tolist())\\n        pred[~fit]=(probabilities[~fit]*w).argmax(1)\\n        fold_deltas.append(macro(y[~fit],pred[~fit])-\\n            macro(y[~fit],probabilities[~fit].argmax(1)))\\n    raw=macro(y,probabilities.argmax(1)); cross=macro(y,pred)\\n    use=bool(cross>=raw+.002 and min(fold_deltas)>=-.002)\\n    return {\'raw_macro_f1\':raw,\'cross_weight_macro_f1\':cross,\\n        \'fold_deltas\':fold_deltas,\'fold_weights\':weights,\'use_weights\':use,\\n        \'weights\':fit_weights(probabilities,y).tolist() if use else [1.,1.,1.,1.],\\n        \'selection_score\':cross if use else raw}\\n\\n\\ndef compare(reference,new_oof,y,folds,configs):\\n    rows=[]\\n    row=cross_weight(reference,y,folds)\\n    row.update(model_index=-1,config=\'v3_reference\',new_model_weight=0.)\\n    rows.append(row)\\n    for i,values in enumerate(new_oof):\\n        for alpha in [.15,.3,.5,1.]:\\n            p=(1-alpha)*reference+alpha*values\\n            row=cross_weight(p,y,folds)\\n            row.update(model_index=i,config=configs[i],new_model_weight=alpha)\\n            rows.append(row)\\n    return sorted(rows,key=lambda row:row[\'selection_score\'],reverse=True)\\n", "train_submit_v2.py": "\\"\\"\\"DS102 v2: đặc trưng quan hệ review/sản phẩm, SVM và calibration F1-macro.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nimport platform\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nimport sklearn\\nfrom scipy import sparse\\nfrom sklearn.metrics import classification_report\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom sklearn.svm import LinearSVC\\nfrom train_submit import load_data, find_data_dir, make_model, make_features, metrics, LABELS, SEED\\nfrom improved_features import prepare, FeaturesV2, nb_ratios\\nfrom select_v2 import macro, calibrate\\n\\n\\n# Cấu hình được chốt từ cross-validation trên train, trước đánh giá dev v2.\\nRECIPE = [\\n    {\'kind\': \'svm\', \'C\': .5, \'weight\': \'balanced\', \'ensemble_weight\': .75},\\n    {\'kind\': \'nb\', \'C\': .5, \'weight\': \'balanced\', \'ensemble_weight\': .25},\\n]\\n\\n\\ndef fit_predict(fit_df, predict_frames, recipe):\\n    \\"\\"\\"Không nhận nhãn của các tập dự đoán.\\"\\"\\"\\n    y = fit_df.spam_label.to_numpy()\\n    features = FeaturesV2()\\n    Xt, Mt = features.fit_transform(fit_df)\\n    transformed = [features.transform(df) for df in predict_frames]\\n    result = [np.zeros((len(df), 4)) for df in predict_frames]\\n    ratios = nb_ratios(Xt, y) if any(c[\'kind\'] == \'nb\' for c in recipe) else None\\n    for config in recipe:\\n        if config[\'kind\'] == \'svm\':\\n            model = make_model(config, y).fit(sparse.hstack([Xt, Mt], format=\'csr\'), y)\\n            if np.max(model.n_iter_) >= model.max_iter:\\n                raise RuntimeError(\'SVM chưa hội tụ; tăng max_iter trước khi xuất kết quả.\')\\n            for j, (Xv, Mv) in enumerate(transformed):\\n                result[j] += config[\'ensemble_weight\'] * model.decision_function(sparse.hstack([Xv, Mv], format=\'csr\'))\\n        elif config[\'kind\'] == \'nb\':\\n            for cls, r in enumerate(ratios):\\n                model = LinearSVC(C=config[\'C\'], class_weight=\'balanced\', dual=True,\\n                                  max_iter=30000, random_state=SEED)\\n                model.fit(sparse.hstack([Xt.multiply(r), Mt], format=\'csr\'), (y == cls).astype(int))\\n                if np.max(model.n_iter_) >= model.max_iter:\\n                    raise RuntimeError(\'NB-SVM chưa hội tụ; tăng max_iter trước khi xuất kết quả.\')\\n                for j, (Xv, Mv) in enumerate(transformed):\\n                    result[j][:, cls] += config[\'ensemble_weight\'] * model.decision_function(\\n                        sparse.hstack([Xv.multiply(r), Mv], format=\'csr\'))\\n        else:\\n            raise ValueError(f\\"Model không được hỗ trợ: {config[\'kind\']}\\")\\n    return result\\n\\n\\ndef main_v2(argv=None):\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'--data-dir\', default=None)\\n    parser.add_argument(\'--output-dir\', default=\'output_v2\')\\n    parser.add_argument(\'--folds\', type=int, default=3)\\n    args = parser.parse_args(argv)\\n    if args.folds < 2:\\n        parser.error(\'--folds phải >= 2\')\\n    data_dir = find_data_dir(args.data_dir)\\n    data = load_data(data_dir)\\n    expected = {\'comment\', \'product_name\', \'description\', \'category\', \'categories\', \'rating\', \'num_sold\', \'num_review\'}\\n    for name in [\'train\', \'dev\', \'test\']:\\n        missing = expected - set(data[name].columns)\\n        if missing:\\n            raise ValueError(f\'{name} thiếu cột {missing}\')\\n        for col in [\'rating\', \'num_sold\', \'num_review\']:\\n            values = pd.to_numeric(data[name][col], errors=\'raise\')\\n            if not np.isfinite(values).all() or (values < 0).any():\\n                raise ValueError(f\'{name}.{col}: giá trị số không hợp lệ\')\\n            data[name][col] = values\\n    train, dev, test = [prepare(data[n]) for n in [\'train\', \'dev\', \'test\']]\\n    output = Path(args.output_dir).expanduser().resolve()\\n    output.mkdir(parents=True, exist_ok=True)\\n    y = train.spam_label.to_numpy()\\n    oof = np.zeros((len(train), 4))\\n    fold_ids = np.full(len(train), -1)\\n    splitter = StratifiedGroupKFold(n_splits=args.folds, shuffle=True, random_state=SEED)\\n    for fold, (tr, va) in enumerate(splitter.split(train, y, groups=train.comment)):\\n        if set(y[tr]) != set(LABELS):\\n            raise ValueError(\'Fold không đủ 4 lớp; giảm số folds.\')\\n        print(f\'CV fold {fold+1}/{args.folds}\', flush=True)\\n        oof[va] = fit_predict(train.iloc[tr], [train.iloc[va].drop(columns=\'spam_label\')], RECIPE)[0]\\n        fold_ids[va] = fold\\n        print(f\'  Raw F1-macro={macro(y[va], oof[va].argmax(axis=1)):.5f}\', flush=True)\\n    cross_pred = np.zeros(len(train), dtype=int)\\n    for f in range(args.folds):\\n        fit = fold_ids != f\\n        bias_fold = calibrate(oof[fit], y[fit])\\n        cross_pred[~fit] = (oof[~fit] + bias_fold).argmax(axis=1)\\n    raw_score = macro(y, oof.argmax(axis=1))\\n    cross_score = macro(y, cross_pred)\\n    use_bias = cross_score > raw_score + .002\\n    bias = calibrate(oof, y) if use_bias else np.zeros(4)\\n    print(f\'OOF raw={raw_score:.5f}; cross-calibrated={cross_score:.5f}; bias={bias.tolist()}\', flush=True)\\n    print(\'Fit toàn bộ train và dự đoán dev/test...\', flush=True)\\n    dev_scores, test_scores = fit_predict(train, [dev.drop(columns=\'spam_label\'), test], RECIPE)\\n    dev_pred = (dev_scores + bias).argmax(axis=1)\\n    test_pred = (test_scores + bias).argmax(axis=1)\\n    dev_y = dev.spam_label.to_numpy()\\n    unique_dev = ~dev.comment.isin(set(train.comment)).to_numpy()\\n    dev_report = metrics(dev_y, dev_pred)\\n    print(f\\"Dev F1-macro={dev_report[\'macro_f1\']:.5f}\\", flush=True)\\n    print(classification_report(dev_y, dev_pred, labels=LABELS, digits=4, zero_division=0), flush=True)\\n    by_id = pd.Series(test_pred, index=test.id.astype(str))\\n    submission = data[\'samplesubmission\'].copy()\\n    submission[\'spam_label\'] = submission.ID.astype(str).map(by_id)\\n    if submission.spam_label.isna().any() or not submission.spam_label.isin(LABELS).all():\\n        raise ValueError(\'Nhãn submission không hợp lệ\')\\n    submission[\'spam_label\'] = submission.spam_label.astype(int)\\n    path = output/\'submission_v2.csv\'\\n    submission.to_csv(path, index=False)\\n    saved = pd.read_csv(path)\\n    pd.testing.assert_frame_equal(saved, submission, check_dtype=False)\\n    if len(saved) != len(test) or not saved.ID.equals(data[\'samplesubmission\'].ID):\\n        raise ValueError(\'Sai số dòng hoặc thứ tự ID\')\\n    pd.DataFrame({\'comment\': data[\'dev\'].comment, \'true_label\': dev_y,\\n                  \'predicted_label\': dev_pred, \'comment_seen_in_train\': ~unique_dev}).to_csv(output/\'dev_predictions_v2.csv\', index=False)\\n    np.savez_compressed(output/\'scores_v2.npz\', oof=oof, y=y, folds=fold_ids,\\n                        dev_scores=dev_scores, test_scores=test_scores, bias=bias)\\n    report = {\\n        \'recipe\': RECIPE, \'seed\': SEED, \'folds\': args.folds, \'python\': platform.python_version(),\\n        \'sklearn\': sklearn.__version__, \'data_sha256\': {name: hashlib.sha256((data_dir/file).read_bytes()).hexdigest()\\n            for name, file in [(\'train\',\'ds102_train.csv\'), (\'dev\',\'ds102_dev.csv\'), (\'test\',\'ds102_test.csv\')]},\\n        \'fit_data\': \'train only; no dev or test used to fit features, models, or calibration\',\\n        \'oof_raw_macro_f1\': raw_score, \'oof_cross_calibration_macro_f1\': cross_score,\\n        \'calibration_bias\': bias.tolist(), \'dev\': dev_report,\\n        \'dev_without_comment_overlap\': metrics(dev_y[unique_dev], dev_pred[unique_dev]) if unique_dev.any() else None,\\n        \'selection_caveat\': \'CV estimates were used for model selection; dev was previously used to inspect baseline errors.\',\\n        \'submission_rows_verified\': len(saved),\\n        \'prediction_counts\': {int(k): int(v) for k, v in saved.spam_label.value_counts().items()},\\n    }\\n    (output/\'metrics_v2.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\\n    print(f\'Đã kiểm tra và lưu {len(saved)} dòng: {path}\', flush=True)\\n    return report\\n\\n\\nif __name__ == \'__main__\':\\n    main_v2()\\n", "catboost_v3.py": "\\"\\"\\"CatBoost text + thông tin review/sản phẩm, không dùng mô hình pretrained.\\"\\"\\"\\nimport numpy as np\\nimport pandas as pd\\nfrom catboost import CatBoostClassifier\\nfrom improved_features import dense_features, tokens\\nfrom train_submit import SEED\\n\\n\\ndef cat_features(df):\\n    a = dense_features(df)\\n    x = pd.DataFrame(a, columns=[f\'stat_{i}\' for i in range(a.shape[1])])\\n    x[\'comment\'] = [\' \'.join(tokens(t)) or \'emptyreview\' for t in df.comment]\\n    x[\'category\'] = df.category.astype(str).to_numpy()\\n    x[\'rating\'] = df.rating.astype(str).to_numpy()\\n    return x\\n\\n\\ndef make_cat(weight=\'SqrtBalanced\', iterations=500):\\n    return CatBoostClassifier(\\n        iterations=iterations, depth=5, learning_rate=.1, l2_leaf_reg=5,\\n        loss_function=\'MultiClass\', auto_class_weights=weight,\\n        thread_count=4, random_seed=SEED, allow_writing_files=False,\\n        text_features=[\'comment\'], cat_features=[\'category\', \'rating\'],\\n        tokenizers=[{\'tokenizer_id\':\'Space\', \'separator_type\':\'ByDelimiter\', \'delimiter\':\' \'}],\\n        dictionaries=[{\'dictionary_id\':\'Word\', \'gram_order\':\'1\', \'occurrence_lower_bound\':\'3\', \'max_dictionary_size\':\'40000\'},\\n                      {\'dictionary_id\':\'Bigram\', \'gram_order\':\'2\', \'occurrence_lower_bound\':\'3\', \'max_dictionary_size\':\'40000\'}],\\n        feature_calcers=[\'BoW:top_tokens_count=700\', \'NaiveBayes\', \'BM25\'],\\n        verbose=False,\\n    )\\n", "experiment_v6.py": "\\"\\"\\"Hosted Colab: template neighbours and probability-calibrated kernel SVM.\\n\\nNo pretrained models or outside data. All feature fitting is inside the same\\ngrouped train folds as previous versions. Dev labels are not read for selection.\\n\\"\\"\\"\\nimport gc,json,time,traceback,os,platform\\nfrom pathlib import Path\\nimport numpy as np\\nfrom sklearn.preprocessing import normalize\\nfrom sklearn.svm import SVC\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom sklearn.metrics.pairwise import rbf_kernel\\nfrom improved_features import prepare,FeaturesV2\\nfrom train_submit import load_data,metrics,SEED\\nfrom select_v2 import macro\\nfrom selection_v5 import cross_weight\\n\\n\\ndef hosted_guard():\\n    if platform.system()!=\'Linux\' or not os.environ.get(\'COLAB_RELEASE_TAG\'):\\n        raise RuntimeError(\'Hosted Google Colab required; local training disabled.\')\\n\\n\\ndef kernel(text_a,meta_a,text_b,meta_b):\\n    k=(text_a@text_b.T).toarray()\\n    k+=1;np.square(k,out=k)\\n    k+=.5*rbf_kernel(meta_a,meta_b,gamma=.3)\\n    k/=4.5\\n    return np.ascontiguousarray(k,dtype=np.float64)\\n\\n\\ndef neighbours(query,train,y,batch_size=256):\\n    \\"\\"\\"Eight nearest TF-IDF reviews; all references belong to fitting data.\\"\\"\\"\\n    counts=np.bincount(y,minlength=4)\\n    class_weight=np.sqrt(len(y)/(4*counts))\\n    probs=np.zeros((query.shape[0],4),dtype=np.float32)\\n    maximum=np.zeros(query.shape[0],dtype=np.float32)\\n    k=min(8,len(y))\\n    for start in range(0,query.shape[0],batch_size):\\n        sim=(query[start:start+batch_size]@train.T).toarray()\\n        idx=np.argpartition(sim,-k,axis=1)[:,-k:]\\n        values=np.take_along_axis(sim,idx,axis=1)\\n        weights=np.maximum(values-.2,0)**4\\n        labels=y[idx]\\n        votes=np.stack([(weights*(labels==c)).sum(axis=1)*class_weight[c] for c in range(4)],axis=1)\\n        votes+=1e-8\\n        probs[start:start+len(sim)]=votes/votes.sum(axis=1,keepdims=True)\\n        maximum[start:start+len(sim)]=values.max(axis=1)\\n    return probs,maximum\\n\\n\\ndef make_kernel_model(y):\\n    counts=np.bincount(y,minlength=4)\\n    cw={c:float(np.sqrt(len(y)/(4*counts[c]))) for c in range(4)}\\n    # libsvm fits its sigmoid calibrators using only the fit partition.\\n    # Outer grouped CV keeps every held-out review group outside that process.\\n    return SVC(C=1.,kernel=\'precomputed\',class_weight=cw,probability=True,\\n               random_state=SEED,cache_size=512)\\n\\n\\ndef neighbour_class_balance(probabilities,counts,power=.5):\\n    \\"\\"\\"Change neighbour voting\'s class-count correction without refitting IDF.\\"\\"\\"\\n    factor=(np.asarray(counts,dtype=float)/max(counts))**(.5-power)\\n    p=probabilities*factor\\n    return p/p.sum(axis=1,keepdims=True)\\n\\n\\ndef candidates(base,hard,prob,near,similarity):\\n    yield \'v3_reference\',base,{\'kind\':\'reference\',\'alpha\':0.}\\n    for alpha in [.05,.1,.2]:\\n        yield f\'hard_kernel_{alpha}\',(1-alpha)*base+alpha*np.eye(4)[hard],{\'kind\':\'hard\',\'alpha\':alpha}\\n    for alpha in [.15,.3,.5]:\\n        yield f\'probability_kernel_{alpha}\',(1-alpha)*base+alpha*prob,{\'kind\':\'probability\',\'alpha\':alpha}\\n    for threshold in [.7,.85]:\\n        for alpha in [.3,.5]:\\n            gate=(similarity>=threshold).astype(float)[:,None]*alpha\\n            yield f\'neighbours_{threshold}_{alpha}\',(1-gate)*base+gate*near,{\'kind\':\'neighbours\',\'alpha\':alpha,\'threshold\':threshold}\\n\\n\\ndef select(base,hard,prob,near,similarity,y,folds):\\n    rows=[]\\n    configurations=list(candidates(base,hard,prob,near,similarity))\\n    # The original inverse-square-root class correction is compared with\\n    # milder/no correction, using only fit-fold label counts.\\n    for power in [0.,.25]:\\n        adjusted=np.empty_like(near)\\n        for fold in np.unique(folds):\\n            mask=folds==fold;counts=np.bincount(y[~mask],minlength=4)\\n            adjusted[mask]=neighbour_class_balance(near[mask],counts,power)\\n        gate=.3*(similarity>=.85).astype(float)[:,None]\\n        configurations.append((f\'neighbours_0.85_0.3_power_{power}\',(1-gate)*base+gate*adjusted,\\n            {\'kind\':\'neighbours\',\'alpha\':.3,\'threshold\':.85,\'class_power\':power}))\\n    for name,p,recipe in configurations:\\n        row=cross_weight(p,y,folds)\\n        raw_folds=[macro(y[folds==f],p[folds==f].argmax(1)) for f in np.unique(folds)]\\n        row[\'fold_macro_f1\']=[v+d for v,d in zip(raw_folds,row[\'fold_deltas\'])] if row[\'use_weights\'] else raw_folds\\n        row.update(name=name,recipe=recipe)\\n        rows.append(row)\\n    rows.sort(key=lambda r:r[\'selection_score\'],reverse=True)\\n    baseline=next(r for r in rows if r[\'recipe\'][\'kind\']==\'reference\')\\n    for row in rows:\\n        row[\'fold_delta_vs_v5\']=(np.array(row[\'fold_macro_f1\'])-np.array(baseline[\'fold_macro_f1\'])).tolist()\\n    return rows\\n\\n\\ndef choose_stable(rows):\\n    \\"\\"\\"Prefer consistent fold gains when aggregate CV differs by < .001.\\"\\"\\"\\n    baseline=next(r for r in rows if r[\'recipe\'][\'kind\']==\'reference\')\\n    best=rows[0]\\n    if best[\'selection_score\']<baseline[\'selection_score\']+.002:return baseline\\n    stable=[r for r in rows if r[\'recipe\'][\'kind\']!=\'reference\'\\n            and r[\'selection_score\']>=best[\'selection_score\']-.001\\n            and min(r[\'fold_delta_vs_v5\'])>=0]\\n    return max(stable,key=lambda r:r[\'selection_score\']) if stable else best\\n\\n\\ndef main(data_dir=\'/content/ds102_data\',output_dir=\'/content/ds102_v6_experiment\',\\n         reference_path=\'/content/colab_reference.npz\'):\\n    hosted_guard()\\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\\n    def status(stage,**values):\\n        (out/\'status.json\').write_text(json.dumps(dict(stage=stage,time=time.time(),**values),indent=2))\\n    try:\\n        df=prepare(load_data(Path(data_dir))[\'train\']);y=df.spam_label.to_numpy()\\n        ref=np.load(reference_path);assert np.array_equal(y,ref[\'y\'])\\n        hard=np.full(len(y),-1,dtype=int);prob=np.zeros((len(y),4),dtype=np.float32)\\n        near=np.zeros_like(prob);similarity=np.zeros(len(y),dtype=np.float32)\\n        folds=np.full(len(y),-1);results=[]\\n        split=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED)\\n        for fold,(tr,va) in enumerate(split.split(df,y,groups=df.comment)):\\n            folds[va]=fold;status(\'features\',fold=fold+1,results=results)\\n            f=FeaturesV2();xt,mt=f.fit_transform(df.iloc[tr])\\n            xv,mv=f.transform(df.iloc[va].drop(columns=\'spam_label\'))\\n            xt=normalize(xt);xv=normalize(xv)\\n            mt.data=np.clip(mt.data,-.75,.75);mv.data=np.clip(mv.data,-.75,.75)\\n            status(\'neighbours\',fold=fold+1,results=results)\\n            near[va],similarity[va]=neighbours(xv,xt,y[tr])\\n            status(\'kernel_matrix\',fold=fold+1,results=results)\\n            kt=kernel(xt,mt,xt,mt);kv=kernel(xv,mv,xt,mt)\\n            status(\'kernel_probability_fit\',fold=fold+1,results=results)\\n            m=make_kernel_model(y[tr]);m.fit(kt,y[tr])\\n            assert np.array_equal(m.classes_,np.arange(4))\\n            hard[va]=m.predict(kv);prob[va]=m.predict_proba(kv)\\n            results.append({\'kernel_hard\':macro(y[va],hard[va]),\\n                            \'kernel_prob\':macro(y[va],prob[va].argmax(1)),\\n                            \'neighbours\':macro(y[va],near[va].argmax(1))})\\n            del m,kt,kv,xt,xv,mt,mv,f;gc.collect()\\n            np.savez_compressed(out/\'partial.npz\',hard=hard,prob=prob,near=near,\\n                                similarity=similarity,y=y,folds=folds)\\n        assert np.array_equal(folds,ref[\'folds\'])\\n        np.savez_compressed(out/\'oof.npz\',hard=hard,prob=prob,near=near,\\n                            similarity=similarity,y=y,folds=folds)\\n        rows=select(ref[\'oof\'],hard,prob,near,similarity,y,folds)\\n        (out/\'selection.json\').write_text(json.dumps(rows,indent=2))\\n        report={\'fold_scores\':results,\'best\':rows[0],\'chosen\':choose_stable(rows),\\n            \'selection_rule\':\'Require >=.002 CV gain; within .001 of best, prefer a candidate improving every fold over v5.\',\\n            \'reference_v5\':next(r for r in rows if r[\'recipe\'][\'kind\']==\'reference\'),\\n            \'pretrained\':False,\'fit_data\':\'train only; dev not read for selection\',\\n            \'execution\':\'hosted Colab CPU\',\\n            \'caveat\':\'Repeated OOF model selection; weight calibration on existing OOF is not fully nested CV.\'}\\n        (out/\'cv_results.json\').write_text(json.dumps(report,indent=2));status(\'complete\',**report)\\n        return report\\n    except BaseException as exc:\\n        status(\'error\',error=str(exc),traceback=traceback.format_exc());raise\\n", "latent_v6.py": "\\"\\"\\"Train-only latent TF-IDF review/product relations + numeric CatBoost.\\n\\nTruncatedSVD is fitted from scratch inside each training fold. No pretrained\\nembeddings, language models or outside corpora are used.\\n\\"\\"\\"\\nimport gc,json,time,traceback\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom scipy import sparse\\nfrom sklearn.decomposition import TruncatedSVD\\nfrom sklearn.preprocessing import normalize\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom catboost import CatBoostClassifier\\nfrom improved_features import FeaturesV2,dense_features,prepare\\nfrom experiment_v6 import hosted_guard\\nfrom selection_v5 import cross_weight\\nfrom select_v2 import macro\\nfrom train_submit import load_data,SEED\\n\\n\\nclass LatentFeatures:\\n    def __init__(self):\\n        self.base=FeaturesV2()\\n        self.svd=TruncatedSVD(n_components=128,n_iter=5,random_state=SEED)\\n\\n    def transform(self,df,fit=False):\\n        if fit:\\n            review=sparse.hstack([self.base.word.fit_transform(df.comment),\\n                                  self.base.char.fit_transform(df.comment)],format=\'csr\')\\n            review=normalize(review)\\n            z=self.svd.fit_transform(review)\\n        else:\\n            review=sparse.hstack([self.base.word.transform(df.comment),\\n                                  self.base.char.transform(df.comment)],format=\'csr\')\\n            review=normalize(review)\\n            z=self.svd.transform(review)\\n        title=normalize(sparse.hstack([self.base.word.transform(df.product_name),\\n                                      self.base.char.transform(df.product_name)],format=\'csr\'))\\n        t=self.svd.transform(title)\\n        cosine=np.asarray(review.multiply(title).sum(axis=1)).ravel()\\n        x=np.hstack([z,t,z*t,np.abs(z-t),cosine[:,None],dense_features(df)])\\n        x=pd.DataFrame(x.astype(np.float32),columns=[f\'f_{i}\' for i in range(x.shape[1])])\\n        x[\'category\']=df.category.astype(str).to_numpy()\\n        x[\'rating\']=df.rating.astype(str).to_numpy()\\n        return x\\n\\n\\ndef make_model():\\n    return CatBoostClassifier(iterations=800,depth=5,learning_rate=.06,l2_leaf_reg=8,\\n        loss_function=\'MultiClass\',auto_class_weights=\'SqrtBalanced\',\\n        cat_features=[\'category\',\'rating\'],max_ctr_complexity=1,\\n        random_seed=SEED,thread_count=2,allow_writing_files=False,verbose=False)\\n\\n\\ndef main(data_dir=\'/content/ds102_data\',output_dir=\'/content/ds102_v6_latent\',reference_path=\'/content/colab_reference.npz\'):\\n    hosted_guard()\\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\\n    def status(stage,**values):\\n        (out/\'status.json\').write_text(json.dumps(dict(stage=stage,time=time.time(),**values),indent=2))\\n    try:\\n        df=prepare(load_data(Path(data_dir))[\'train\']);y=df.spam_label.to_numpy()\\n        ref=np.load(reference_path);assert np.array_equal(y,ref[\'y\'])\\n        trees=[400,800];oof=np.zeros((2,len(y),4),dtype=np.float32)\\n        folds=np.full(len(y),-1);results=[[],[]]\\n        split=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED)\\n        for f,(tr,va) in enumerate(split.split(df,y,groups=df.comment)):\\n            folds[va]=f;status(\'fit_svd\',fold=f+1,results=results)\\n            features=LatentFeatures();xt=features.transform(df.iloc[tr],fit=True)\\n            xv=features.transform(df.iloc[va].drop(columns=\'spam_label\'))\\n            status(\'fit_catboost\',fold=f+1,results=results)\\n            m=make_model();m.fit(xt,y[tr],verbose=False)\\n            for i,n in enumerate(trees):\\n                oof[i,va]=m.predict_proba(xv,ntree_end=n)\\n                results[i].append(macro(y[va],oof[i,va].argmax(1)))\\n            del features,m,xt,xv;gc.collect()\\n            np.savez_compressed(out/\'partial.npz\',oof=oof,y=y,folds=folds)\\n        assert np.array_equal(folds,ref[\'folds\'])\\n        rows=[]\\n        for i,n in enumerate(trees):\\n            for alpha in [.15,.3,.5]:\\n                p=(1-alpha)*ref[\'oof\']+alpha*oof[i]\\n                row=cross_weight(p,y,folds)\\n                raw=[macro(y[folds==f],p[folds==f].argmax(1)) for f in np.unique(folds)]\\n                row[\'fold_macro_f1\']=[a+b for a,b in zip(raw,row[\'fold_deltas\'])] if row[\'use_weights\'] else raw\\n                row.update(name=f\'latent_cat_{n}_{alpha}\',recipe={\'kind\':\'latent\',\'trees\':n,\'alpha\':alpha})\\n                rows.append(row)\\n        rows.sort(key=lambda r:r[\'selection_score\'],reverse=True)\\n        np.savez_compressed(out/\'oof.npz\',oof=oof,y=y,folds=folds)\\n        (out/\'selection.json\').write_text(json.dumps(rows,indent=2))\\n        report={\'best\':rows[0],\'fold_scores\':results,\'pretrained\':False,\'fit_data\':\'train only\',\\n                \'representation\':\'TF-IDF and 64D SVD fitted on each training fold; review/title relations\',\\n                \'caveat\':\'Repeated OOF selection; weighting checks not fully nested CV.\'}\\n        (out/\'cv_results.json\').write_text(json.dumps(report,indent=2));status(\'complete\',**report)\\n        return report\\n    except BaseException as exc:\\n        status(\'error\',error=str(exc),traceback=traceback.format_exc());raise\\n", "run_submission.py": "import json,os,platform\\nfrom pathlib import Path\\nassert platform.system()==\'Linux\' and os.environ.get(\'COLAB_RELEASE_TAG\'),\'Hosted Colab required\'\\nfrom train_submit_v27_experimental import main\\nselection={\'recipe\':{\'kind\':\'latent\',\'trees\':800,\'alpha\':.3},\'weights\':[1.,1.2,1.2,.9],\\n           \'name\':\'V27_experimental_class2_text2000_update\'}\\nout=Path(\'/content/ds102_v27_experimental\')\\nreport=main(selection,data_dir=\'/content/ds102_v27_data\',output_dir=str(out),reference_path=None)\\nreport.pop(\'v5_reference\',None)\\nreport[\'baseline_user_reported_kaggle\']=0.77436\\nreport[\'cv_evidence\']={\'baseline_oof\': {\'macro_f1\': 0.7749451430616494, \'accuracy\': 0.8978051167342374, \'classification_report\': {\'0\': {\'precision\': 0.9235403151065802, \'recall\': 0.9479642313546424, \'f1-score\': 0.9355929020749225, \'support\': 10512.0}, \'1\': {\'precision\': 0.6908212560386473, \'recall\': 0.6777251184834123, \'f1-score\': 0.6842105263157895, \'support\': 211.0}, \'2\': {\'precision\': 0.628968253968254, \'recall\': 0.5747960108794198, \'f1-score\': 0.600663192799621, \'support\': 1103.0}, \'3\': {\'precision\': 0.913515862668405, \'recall\': 0.8475806451612903, \'f1-score\': 0.8793139510562644, \'support\': 2480.0}, \'accuracy\': 0.8978051167342374, \'macro avg\': {\'precision\': 0.7892114219454717, \'recall\': 0.7620165014696912, \'f1-score\': 0.7749451430616494, \'support\': 14306.0}, \'weighted avg\': {\'precision\': 0.8956584930077698, \'recall\': 0.8978051167342374, \'f1-score\': 0.8963058232868542, \'support\': 14306.0}}, \'confusion_matrix\': [[9965, 47, 338, 162], [54, 143, 3, 11], [443, 0, 634, 26], [328, 17, 33, 2102]]}, \'candidate_oof\': {\'macro_f1\': 0.7758703545309641, \'accuracy\': 0.8980148189570809, \'classification_report\': {\'0\': {\'precision\': 0.9236973854997219, \'recall\': 0.9477739726027398, \'f1-score\': 0.9355808057094563, \'support\': 10512.0}, \'1\': {\'precision\': 0.6908212560386473, \'recall\': 0.6777251184834123, \'f1-score\': 0.6842105263157895, \'support\': 211.0}, \'2\': {\'precision\': 0.6299212598425197, \'recall\': 0.5802357207615594, \'f1-score\': 0.6040585181689476, \'support\': 1103.0}, \'3\': {\'precision\': 0.9146713104048759, \'recall\': 0.8471774193548387, \'f1-score\': 0.879631567929663, \'support\': 2480.0}, \'accuracy\': 0.8980148189570809, \'macro avg\': {\'precision\': 0.7897778029464412, \'recall\': 0.7632280578006375, \'f1-score\': 0.7758703545309641, \'support\': 14306.0}, \'weighted avg\': {\'precision\': 0.8960476863419281, \'recall\': 0.8980148189570809, \'f1-score\': 0.8966137763649065, \'support\': 14306.0}}, \'confusion_matrix\': [[9963, 47, 340, 162], [54, 143, 3, 11], [440, 0, 640, 23], [329, 17, 33, 2101]]}, \'delta\': 0.0009252114693146796, \'fold_deltas\': [0.003064039874932978, -0.0010316370313683265, 0.0007023404137642686], \'novel_fake_f1_delta\': 0.0, \'gates\': {\'macro\': False, \'folds\': True, \'worst_fold\': True, \'novel\': True}, \'bootstrap\': {\'observed_delta\': 0.0009252114693146796, \'mean\': 0.0008920241482336649, \'sd\': 0.0008114801694280067, \'q025\': -0.0005749045169613165, \'q05\': -0.0003945911612598274, \'q95\': 0.0022830441994839685, \'q975\': 0.002569286549046811, \'fraction_delta_gt_0\': 0.862, \'groups\': 13656, \'rows\': 14306, \'resamples\': 2000, \'seed\': 102, \'fraction_resamples_missing_a_class\': 0.0, \'intervals\': \'q025-q975: central 95%; q05-q95: central 90%\', \'caveat\': \'Conditional on these selected/fitted predictions. Does not remove train/validation template overlap, selection bias, or training-sample uncertainty. Positive fraction is not a posterior probability or a calibrated p-value.\'}}\\nreport[\'qualification\']=\'Experimental: OOF improved but the original +0.001 macro threshold failed; original rejection preserved. Kaggle score unknown.\'\\nreport[\'recipe\']=\'Start with SVD128 baseline; delta=clip(0.175*(new_cat2-old_cat2),-baseline2,baseline0); add delta to class2, subtract from class0.\'\\nreport[\'inference_note\']=\'Fresh fit of all components from train CSV. Earlier exploratory CSV used authorized baseline caches. Numerical differences are possible. Dev excluded from training.\'\\n(out/\'metrics_v27_experimental.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2))\\nprint(\'EXPERIMENTAL CSV:\',out/\'submission_v27_experimental.csv\')\\nprint(\'Dev F1:\',report[\'dev\'][\'macro_f1\'])\\n", "train_submit_v27_experimental.py": "\\"\\"\\"Train-only DS102 V27 exploratory class2 text inference; hosted Colab required; no pretrained.\\"\\"\\"\\nimport gc,hashlib,json,time,traceback\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom scipy.special import softmax\\nfrom sklearn.preprocessing import normalize\\nfrom improved_features import prepare,FeaturesV2\\nfrom experiment_v6 import hosted_guard,kernel,neighbours,make_kernel_model,neighbour_class_balance\\nfrom train_submit import load_data,metrics\\n\\n\\ndef baseline_predict(train,frames):\\n    from train_submit_v2 import fit_predict,RECIPE\\n    from catboost_v3 import make_cat,cat_features\\n    margins=fit_predict(train,frames,RECIPE);gc.collect()\\n    m=make_cat(\'SqrtBalanced\',iterations=500);m.set_params(thread_count=2)\\n    m.fit(cat_features(train),train.spam_label.to_numpy(),verbose=250)\\n    old_cat=[m.predict_proba(cat_features(d)) for d in frames]\\n    del m;gc.collect()\\n    m=make_cat(\'SqrtBalanced\',iterations=500)\\n    m.set_params(thread_count=2,feature_calcers=[\'BoW:top_tokens_count=2000\',\'NaiveBayes\',\'BM25\'])\\n    m.fit(cat_features(train),train.spam_label.to_numpy(),verbose=100)\\n    return [(.5*softmax(3*s,axis=1)+.5*p,p,m.predict_proba(cat_features(d)))\\n            for s,p,d in zip(margins,old_cat,frames)]\\n\\n\\ndef component_predict(train,frames,recipe,status=None):\\n    hosted_guard()\\n    if recipe[\'kind\']==\'latent\':\\n        from latent_v6 import LatentFeatures,make_model as make_latent_model\\n        if status:status(\'fit_latent_features\')\\n        f=LatentFeatures();f.svd.set_params(n_components=128,random_state=102);xt=f.transform(train,fit=True)\\n        if status:status(\'fit_latent_catboost\')\\n        m=make_latent_model();m.fit(xt,train.spam_label.to_numpy(),verbose=False)\\n        return [(m.predict_proba(f.transform(df),ntree_end=recipe[\'trees\']),None) for df in frames]\\n    f=FeaturesV2();xt,mt=f.fit_transform(train)\\n    xt=normalize(xt);mt.data=np.clip(mt.data,-.75,.75)\\n    y=train.spam_label.to_numpy();kind=recipe[\'kind\']\\n    if kind==\'neighbours\':\\n        result=[]\\n        for df in frames:\\n            x,_=f.transform(df);p,s=neighbours(normalize(x),xt,y)\\n            if recipe.get(\'class_power\',.5)!=.5:\\n                p=neighbour_class_balance(p,np.bincount(y,minlength=4),recipe[\'class_power\'])\\n            result.append((p,s))\\n        return result\\n    if status:status(\'full_kernel\')\\n    gram=kernel(xt,mt,xt,mt)\\n    m=make_kernel_model(y)\\n    if kind==\'hard\':m.set_params(probability=False)\\n    if status:status(\'full_kernel_fit\')\\n    m.fit(gram,y);del gram;gc.collect()\\n    result=[]\\n    for frame_idx,df in enumerate(frames):\\n        if status:status(\'predict\',frame=frame_idx)\\n        pieces=[]\\n        for start in range(0,len(df),256):\\n            xv,mv=f.transform(df.iloc[start:start+256]);xv=normalize(xv)\\n            mv.data=np.clip(mv.data,-.75,.75)\\n            gram=kernel(xv,mv,xt,mt)\\n            pieces.append(np.eye(4)[m.predict(gram)] if kind==\'hard\' else m.predict_proba(gram))\\n        result.append((np.vstack(pieces),None))\\n    return result\\n\\n\\ndef main(selection,data_dir=\'/content/ds102_data\',output_dir=\'/content/ds102_v6\',reference_path=None):\\n    hosted_guard()\\n    assert reference_path is None,\'This runner fits every component fresh; no reference cache.\'\\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\\n    def status(stage,**values):\\n        (out/\'status.json\').write_text(json.dumps(dict(stage=stage,time=time.time(),**values),indent=2))\\n    try:\\n        data=load_data(Path(data_dir))\\n        train,dev,test=[prepare(data[k]) for k in [\'train\',\'dev\',\'test\']]\\n        frames=[dev.drop(columns=\'spam_label\'),test]\\n        recipe=selection[\'recipe\']\\n        if recipe[\'kind\']==\'reference\':raise ValueError(\'v6 experiment did not choose a new component; retain v5.\')\\n        status(\'component_features\',recipe=recipe)\\n        extra=component_predict(train,frames,recipe,status)\\n        gc.collect();status(\'baseline_predictions\')\\n        if reference_path:\\n            z=np.load(reference_path)\\n            assert np.array_equal(z[\'y\'],train.spam_label.to_numpy())\\n            base=[z[\'dev\'],z[\'test\']]\\n            baseline_source=\'previously user-authorized cached v3 predictions\'\\n        else:\\n            base=baseline_predict(train,frames)\\n            baseline_source=\'fresh train-only fit on hosted Colab\'\\n        scores=[];alpha=float(recipe[\'alpha\'])\\n        for (b,old_cat,new_cat),(p,s) in zip(base,extra):\\n            weight=alpha\\n            if recipe[\'kind\']==\'neighbours\':weight=alpha*(s>=recipe[\'threshold\']).astype(float)[:,None]\\n            combined=(1-weight)*b+weight*p\\n            delta=np.clip(.175*(new_cat[:,2]-old_cat[:,2]),-combined[:,2],combined[:,0])\\n            combined[:,2]+=delta;combined[:,0]-=delta\\n            assert (combined>=0).all() and np.allclose(combined.sum(1),1,atol=1e-5)\\n            scores.append(combined)\\n        weights=np.asarray(selection[\'weights\'])\\n        dp,tp=[(p*weights).argmax(1) for p in scores]\\n        sub=data[\'samplesubmission\'].copy()\\n        lookup=pd.Series(tp,index=test.id.astype(str))\\n        sub[\'spam_label\']=sub.ID.astype(str).map(lookup)\\n        if sub.spam_label.isna().any() or not sub.spam_label.isin([0,1,2,3]).all():raise ValueError(\'Invalid IDs/labels\')\\n        sub.spam_label=sub.spam_label.astype(int)\\n        path=out/\'submission_v27_experimental.csv\';sub.to_csv(path,index=False)\\n        saved=pd.read_csv(path);pd.testing.assert_frame_equal(saved,sub,check_dtype=False)\\n        assert len(saved)==len(test) and saved.ID.equals(data[\'samplesubmission\'].ID)\\n        np.savez_compressed(out/\'predictions.npz\',dev=scores[0],test=scores[1],weights=weights)\\n        report={\'selection\':selection,\'dev\':metrics(dev.spam_label.to_numpy(),dp),\\n            \'v5_reference\':{\'cross_weight_oof_macro_f1\':.7681613908279921,\'dev_macro_f1\':.7348200162618499,\'user_reported_kaggle\':.76740},\\n            \'rows_verified\':len(saved),\'pretrained\':False,\'fit_data\':\'train only; dev/test excluded from feature/model fitting\',\\n            \'selection_data\':\'grouped train OOF for candidate hyperparameters; dev used to compare finalists\',\\n            \'new_training_execution\':\'hosted Google Colab CPU\',\'baseline_source\':baseline_source,\\n            \'libraries\':{\'scikit-learn\':\'1.7.2; BSD-3-Clause\',\'CatBoost\':\'1.2.10; Apache-2.0\'},\\n            \'data_sha256\':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(data_dir).glob(\'ds102_*.csv\')},\\n            \'caveat\':\'Repeated OOF selection; weighting cross-validation is on existing OOF, not fully nested. Dev reused in development; Kaggle score unknown.\'}\\n        (out/\'metrics_v27_experimental.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2))\\n        status(\'complete\',dev_macro_f1=report[\'dev\'][\'macro_f1\'],rows=len(saved))\\n        return report\\n    except BaseException as exc:\\n        status(\'error\',error=str(exc),traceback=traceback.format_exc());raise\\n"}')
for name,content in modules.items():
    (source/name).write_text(content)


In [ ]:
import os,sys,subprocess
env=dict(os.environ,OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2')
subprocess.run([sys.executable,'-u',str(source/'run_submission.py')],cwd=str(source),env=env,check=True)


In [ ]:
from google.colab import files
files.download('/content/ds102_v27_experimental/submission_v27_experimental.csv')
files.download('/content/ds102_v27_experimental/metrics_v27_experimental.json')
